In [1]:
#!pip install docx
#!pip uninstall docx python-docx -y

#!pip install python-docx

#!pip install langchain
#!pip install langchain_google_genai
#!pip uninstall pydantic langchain langchain_google_genai -y
#!pip install pydantic==2.1.1
#!pip install langchain==0.1.16
#!pip install langchain-google-genai==0.0.11
#!pip install chromadb
#!pip install sentence-transformers
#!pip install tf-keras

In [2]:
#%%writefile .env #To create .env file
#GEMINI_API_KEY=actual_api_key

In [3]:
#!dir -a #To check if .env is created

In [5]:
import google.generativeai as genai
import os
from dotenv import load_dotenv

from docx import Document as DocxDocument

import helper_func
from helper_func import chat_with_model, moderation_check

from langchain.embeddings import HuggingFaceEmbeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.schema import Document

In [6]:
load_dotenv()
api_key=os.getenv("GEMINI_API_KEY")
genai.configure(api_key=api_key)

In [7]:
#print(api_key)

In [8]:
doc=DocxDocument('Common bike problems- RAG.docx') #Reading word document

para_data=[] #Document reads plain texts (paragraphs)
table_data=[] #Document reads tables also

for para in doc.paragraphs: #Loops through the texts other than table
    if para.text.strip():
        para_data.append(para.text)

for table in doc.tables: #Loops through each table
    for row in table.rows: #Loops through each row in the table
        row_text=[cell.text.strip() for cell in row.cells] #Loops through each cell in the table & stores the texts of each cell
        table_data.append('|'.join(row_text)) #

documents=[Document(page_content=text) for text in (para_data + table_data)]

In [ ]:
print(para_data)
print(table_data)
print(documents)

In [9]:
from sentence_transformers import SentenceTransformer
SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

In [10]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
db=Chroma.from_documents(documents,embeddings)

In [11]:
retriever = db.as_retriever()

In [12]:
chat_history=[]

In [13]:
#Main function:
def process_user_message(user_text):
    docs = retriever.get_relevant_documents(user_text)
    context = "\n".join([d.page_content for d in docs])

    #1. Moderation check:
    mod_response=moderation_check(user_text)
    if mod_response=='Y':
        print('The moderation response is YES')
        return 'Your message contains inappropriate info, we can not continue.'
    else:
        print('The moderation response is NO')

    #2. Create chatbot response:
    System_instruction='''You are helpful assistant that provides solutions to customers' bike faults, similar to a mechanic. 
    Remember that you are not an exact mechanic, so in-depth knowledge transfer or analysis is not required.
    Read the information provided to you and answer. If required, you can ask follow-up questions and then provide your solution.'''

    message=[
        {'role':'user','parts':[{'text':System_instruction + user_text}]}
    ]

    bot_reply=chat_with_model(message, context, max_tokens=500)

    return bot_reply

In [14]:
print("Chatbot is running. Type 'exit' to stop.\n")

while True:
    user_text = input("You: ")

    if user_text.lower() in ["exit", "quit"]:
        print("Bot: Goodbye!")
        break

    bot_reply = process_user_message(user_text)

    print("Bot response:", bot_reply)
    print()

Chatbot is running. Type 'exit' to stop.



You:  scooty didnt start for the past 2 days


C:\Users\DELL\anaconda3\Lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(


The moderation response is NO
Bot response: To help diagnose the issue, could you please tell me:
1.  Did the engine make any sound or attempt to crank when you tried to start it?
2.  Is there fuel available in the scooty?



You:  yes fuel available, yes, it gave starting sound but didnt start


The moderation response is NO
Bot response: **FINAL DIAGNOSIS:**
Based on your description that the engine makes a starting sound but doesn't fire up, despite having fuel, the most likely issues are either a problem with the **ignition system (no spark or weak spark)** or a **fuel delivery issue (fuel not reaching the engine's combustion chamber)**.

**ACTION:**
To help pinpoint the exact issue, please perform these checks:

1.  **Attempt a Kick Start:** Try starting the bike using the kick starter. If it starts, the problem is likely with your electric start system (e.g., a weak battery not providing enough power for both the starter and a strong spark). If it doesn't start, the issue is more fundamental.
2.  **Check Battery Strength:** Even if the starter motor is turning, a weak battery might not be able to provide enough power for a strong spark. Check if your headlights are bright and the horn sounds strong. If they are dim or weak, the battery might be low.
3.  **Inspect Spark Pl

You:  quit


Bot: Goodbye!
